# 01 – Train your first tiny language model
**Student notebook | Beginner | about 30–40 minutes**

In this activity, you will train a very small text model. It learns which character often comes after another character, then uses those patterns to generate new text.

**Important:** This is a real, very simple *character-level statistical language model*. It is not ChatGPT and it does not understand meaning. Its limitations are part of the experiment.

### How to use this notebook
1. Run the cells from top to bottom using the ▶ button.
2. Choose a training text.
3. Train the model.
4. Enter a starting phrase and generate text.
5. Record what you notice in your worksheet.

No account or software installation is needed when your teacher launches this notebook through Binder.

## 1. Load the training texts
Choose one text collection. The model will learn from this text only.

In [ ]:
from pathlib import Path
from collections import defaultdict, Counter
import random
import matplotlib.pyplot as plt

DATA = {
    "Fairy tales": Path("data/fairy_tales.txt"),
    "Football": Path("data/football.txt"),
    "Science fiction": Path("data/science_fiction.txt"),
}

print("Available training texts:")
for name in DATA:
    print("-", name)

In [ ]:
# Change this value to choose a different training text.
dataset_name = "Fairy tales"

text = DATA[dataset_name].read_text(encoding="utf-8").lower()
print("Selected:", dataset_name)
print("Number of characters:", len(text))
print()
print("A sample from the training text:")
print(text[:350])

## 2. Train the model
The model counts what character follows each character. For example, after the letter `a`, the next character might often be `n`, `r`, or a space.

This is much simpler than a modern LLM, but it lets us see the basic idea of learning patterns from examples.

In [ ]:
def train_character_model(training_text):
    transitions = defaultdict(Counter)
    for current_char, next_char in zip(training_text, training_text[1:]):
        transitions[current_char][next_char] += 1
    return transitions

model = train_character_model(text)
print("Training finished!")
print("Number of different current characters:", len(model))
print("The model has learned", sum(len(v) for v in model.values()),
      "character-transition possibilities.")

## 3. Generate new text
The model starts with your phrase and repeatedly chooses a likely next character based on what it learned.

The `randomness` setting changes how strongly the model prefers common next characters:
- Lower values: usually more predictable.
- Higher values: usually more varied.

This is a simplified sampling control, not exactly the temperature implementation used by GPT.

In [ ]:
def generate_text(start, transitions, length=400, randomness=0.8):
    if not start:
        start = "once upon a time "
    result = start.lower()
    current = result[-1]

    for _ in range(length):
        choices = transitions.get(current)
        if not choices:
            current = random.choice(list(transitions.keys()))
            result += current
            continue

        chars = list(choices.keys())
        counts = list(choices.values())
        weights = [count ** (1 / max(randomness, 0.05)) for count in counts]
        next_char = random.choices(chars, weights=weights, k=1)[0]
        result += next_char
        current = next_char

    return result

# Edit these three settings and run this cell again.
starting_phrase = "once upon a time"
randomness = 0.8
number_of_characters = 400

generated = generate_text(
    starting_phrase, model,
    length=number_of_characters,
    randomness=randomness
)
print(generated)

## 4. Your experiment
Try the same starting phrase with:
- Fairy tales
- Football
- Science fiction

Change only `dataset_name` in section 1, then run the cells again from section 1 onwards.

### Think about it
1. Does the generated text resemble the training text?
2. Does it produce complete, meaningful sentences?
3. What changes when you use a different dataset?
4. Why might the model repeat letters or produce nonsense?

**Remember:** the model has learned local character patterns, not facts or meaning.

In [ ]:
# Optional: inspect the most common next characters after a character.
character_to_inspect = "t"
if character_to_inspect in model:
    most_common = model[character_to_inspect].most_common(8)
    labels = [repr(char) for char, count in most_common]
    values = [count for char, count in most_common]
    plt.bar(labels, values)
    plt.title(f"Characters that follow {character_to_inspect!r}")
    plt.xlabel("Next character")
    plt.ylabel("Count in training text")
    plt.show()
else:
    print("That character does not occur in the training text.")